# Notebook 3: Advanced Spark Concepts for Genomics

**Optional bonus material** for students who finish the main exercise early.

**Topics:**
- Section 1: Window Functions (~30 min)
- Section 2: User-Defined Functions (UDFs) (~20 min)
- Section 3: Partitioning Strategies (~25 min)
- Section 4: Machine Learning Preview (~30 min)

Each section can be completed independently.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import time

spark = SparkSession.builder \
    .appName("BINFX410_Advanced") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

data_dir = "/workspace/data"
if not os.path.exists(data_dir):
    data_dir = os.path.join(os.path.dirname(os.getcwd()), "data")

variants = spark.read.csv(os.path.join(data_dir, "variants_medium.csv"),
                          header=True, inferSchema=True)
genes = spark.read.csv(os.path.join(data_dir, "genes.bed"), sep="\t", header=False,
    schema="chrom STRING, start INT, end INT, gene_name STRING, strand STRING")
samples = spark.read.csv(os.path.join(data_dir, "samples.csv"),
                         header=True, inferSchema=True)

print(f"Variants: {variants.count():,}")
print(f"Genes: {genes.count()}")
print(f"Samples: {samples.count()}")

---
## Section 1: Window Functions

Window functions compute values across a set of rows related to the current row,
**without collapsing the rows** (unlike `groupBy`). They are defined by:
- `partitionBy`: defines groups (like groupBy, but rows aren't collapsed)
- `orderBy`: defines ordering within each group
- Optional frame specification: which rows relative to current row

In [ ]:
# Rank variants by QUAL score within each chromosome
window_qual = Window.partitionBy("CHROM").orderBy(F.col("QUAL").desc())

ranked = variants.withColumn("qual_rank", F.row_number().over(window_qual))

# Top 3 variants per chromosome
top3_per_chrom = ranked.filter(F.col("qual_rank") <= 3)
top3_per_chrom.select("CHROM", "POS", "QUAL", "qual_rank") \
    .orderBy("CHROM", "qual_rank").show(30)

In [ ]:
# Running statistics: cumulative count and mean quality by position
window_pos = Window.partitionBy("CHROM").orderBy("POS") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

with_running = variants.withColumn(
    "running_count", F.count("*").over(window_pos)
).withColumn(
    "running_mean_qual", F.round(F.mean("QUAL").over(window_pos), 2)
)

with_running.filter(F.col("CHROM") == "chr1") \
    .select("CHROM", "POS", "QUAL", "running_count", "running_mean_qual") \
    .orderBy("POS").show(20)

In [ ]:
# Lead/Lag: distance to the next variant on the same chromosome
window_next = Window.partitionBy("CHROM").orderBy("POS")

with_distance = variants.withColumn(
    "next_pos", F.lead("POS").over(window_next)
).withColumn(
    "distance_to_next", F.col("next_pos") - F.col("POS")
)

# Find clustered variants (< 100bp apart)
clustered = with_distance.filter(F.col("distance_to_next") < 100)
print(f"Clustered variant pairs (< 100bp apart): {clustered.count()}")
clustered.select("CHROM", "POS", "next_pos", "distance_to_next") \
    .orderBy("distance_to_next").show(10)

---
## Section 2: User-Defined Functions (UDFs)

UDFs let you run custom Python code on Spark DataFrames. However, they are
**slower than built-in functions** because data must be serialized between the
JVM and Python. Use built-in functions whenever possible.

In [ ]:
# Simple row-at-a-time UDF
@F.udf(StringType())
def classify_variant(ref, alt):
    """Classify variant type based on REF and ALT alleles."""
    if ref is None or alt is None:
        return "unknown"
    if len(ref) == 1 and len(alt) == 1:
        return "SNV"
    elif len(ref) < len(alt):
        return "insertion"
    elif len(ref) > len(alt):
        return "deletion"
    return "MNV"

variants_typed = variants.withColumn(
    "variant_type", classify_variant(F.col("REF"), F.col("ALT"))
)
variants_typed.groupBy("variant_type").count().show()

In [ ]:
# Performance: UDF vs built-in functions

# UDF approach (slower)
start = time.time()
variants.withColumn("vtype", classify_variant(F.col("REF"), F.col("ALT"))).count()
udf_time = time.time() - start

# Built-in approach (faster)
start = time.time()
variants.withColumn(
    "vtype",
    F.when((F.length("REF") == 1) & (F.length("ALT") == 1), "SNV")
     .when(F.length("REF") < F.length("ALT"), "insertion")
     .when(F.length("REF") > F.length("ALT"), "deletion")
     .otherwise("MNV")
).count()
builtin_time = time.time() - start

print(f"UDF time:      {udf_time:.3f}s")
print(f"Built-in time: {builtin_time:.3f}s")
print(f"Built-in is {udf_time/builtin_time:.1f}x faster")

In [ ]:
# Pandas UDF (vectorized - much faster than row UDFs)
from pyspark.sql.functions import pandas_udf

@pandas_udf(StringType())
def classify_variant_vectorized(ref: pd.Series, alt: pd.Series) -> pd.Series:
    """Vectorized variant classification using pandas."""
    result = pd.Series(["unknown"] * len(ref))
    mask_snv = (ref.str.len() == 1) & (alt.str.len() == 1)
    mask_ins = ref.str.len() < alt.str.len()
    mask_del = ref.str.len() > alt.str.len()
    result[mask_snv] = "SNV"
    result[mask_ins] = "insertion"
    result[mask_del] = "deletion"
    result[~mask_snv & ~mask_ins & ~mask_del & ref.notna()] = "MNV"
    return result

start = time.time()
variants.withColumn("vtype", classify_variant_vectorized(F.col("REF"), F.col("ALT"))).count()
pandas_udf_time = time.time() - start

print(f"Row UDF:    {udf_time:.3f}s")
print(f"Pandas UDF: {pandas_udf_time:.3f}s")
print(f"Built-in:   {builtin_time:.3f}s")
print("\nKey takeaway: Always prefer built-in > pandas UDF > row UDF")

---
## Section 3: Partitioning Strategies

In [ ]:
# Hash partitioning by column
hash_part = variants.repartition(8, "CHROM")
print("Hash partitioned by CHROM:")
hash_part.withColumn("pid", F.spark_partition_id()) \
    .groupBy("pid").agg(
        F.count("*").alias("count"),
        F.countDistinct("CHROM").alias("n_chroms")
    ).orderBy("pid").show()

In [ ]:
# Range partitioning
range_part = variants.repartitionByRange(8, "CHROM", "POS")
print("Range partitioned by CHROM, POS:")
range_part.withColumn("pid", F.spark_partition_id()) \
    .groupBy("pid").agg(
        F.count("*").alias("count"),
        F.min("CHROM").alias("min_chrom"),
        F.max("CHROM").alias("max_chrom")
    ).orderBy("pid").show()

In [ ]:
# Impact on query performance
start = time.time()
variants.groupBy("CHROM").count().collect()
default_time = time.time() - start

partitioned = variants.repartition("CHROM")
partitioned.cache()
partitioned.count()

start = time.time()
partitioned.groupBy("CHROM").count().collect()
partitioned_time = time.time() - start

partitioned.unpersist()

print(f"Default partitioning:        {default_time:.3f}s")
print(f"Pre-partitioned by CHROM:    {partitioned_time:.3f}s")

In [ ]:
# Detecting data skew
partition_sizes = variants.withColumn("pid", F.spark_partition_id()) \
    .groupBy("pid").count().orderBy("pid")

partition_sizes.show()

stats = partition_sizes.agg(
    F.min("count").alias("min_rows"),
    F.max("count").alias("max_rows"),
    F.round(F.mean("count"), 0).alias("avg_rows"),
    F.round(F.stddev("count"), 0).alias("stddev_rows")
)
stats.show()
print("High stddev relative to mean indicates data skew.")

---
## Section 4: Machine Learning Preview

A brief introduction to Spark MLlib - building a simple classifier on variant data.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# Create a binary label: is the variant high-quality?
ml_data = variants.withColumn(
    "label",
    F.when(
        (F.col("QUAL") >= 50) & (F.col("FILTER") == "PASS") & (F.col("DP") >= 20),
        1.0
    ).otherwise(0.0)
)

print("Label distribution:")
ml_data.groupBy("label").count().show()

In [ ]:
# Build ML pipeline
consequence_indexer = StringIndexer(
    inputCol="CONSEQUENCE", outputCol="consequence_idx",
    handleInvalid="keep"
)

feature_cols = ["DP", "AF", "GQ", "consequence_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

rf = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=20, maxDepth=5, seed=42
)

pipeline = Pipeline(stages=[consequence_indexer, assembler, rf])

# Train/test split
train, test = ml_data.randomSplit([0.8, 0.2], seed=42)
print(f"Training set: {train.count():,} variants")
print(f"Test set:     {test.count():,} variants")

# Train model
start = time.time()
model = pipeline.fit(train)
print(f"Training time: {time.time() - start:.1f}s")

In [ ]:
# Evaluate
predictions = model.transform(test)

evaluator = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator.evaluate(predictions)
print(f"AUC-ROC: {auc:.3f}")

# Confusion matrix summary
predictions.groupBy("label", "prediction").count() \
    .orderBy("label", "prediction").show()

# Feature importance
rf_model = model.stages[-1]
importances = list(zip(feature_cols, rf_model.featureImportances.toArray()))
print("\nFeature Importances:")
for feat, imp in sorted(importances, key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.3f}")

This is just a preview! Real variant pathogenicity prediction uses many more features
(conservation scores, protein structure, population databases) and more sophisticated
models. Tools like CADD and REVEL use similar ML approaches at genome scale.

---
## Summary

- **Window functions** enable analytics without losing row-level detail
- **UDFs** provide flexibility but prefer built-in functions for performance
- **Partitioning strategy** significantly affects query performance
- **Spark MLlib** enables ML on datasets too large for scikit-learn

In [ ]:
spark.stop()
print("Advanced concepts notebook complete!")